# NB_SLV_PN_HOTEL_ROOM_OFFER

RECOMPUTE `lh_vv_silver.dbo.slv_pn_hotel_room_offer`. Một dòng mỗi `room_type_id`. Không gắn rate plan và không tạo giá.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
FLAG = "_is_deleted"
OUT_COLS = (
    "room_type_id", "status", "available_rooms", "max_occupancy", "adult_capacity", "child_capacity",
    "view_type", "deleted", "is_bookable",
)
ROOM_TYPES = {
    "id": "string", "status": "string", "available_rooms": "int", "max_occupancy": "int",
    "adult_capacity": "int", "child_capacity": "int", "view_type": "string", "deleted": "boolean",
}

def _require_types(df, expected, name):
    """Chặn khi build thiếu cột hoặc kiểu lệch Field Mapping L1. Không cast.

    Input : df — DataFrame L1; expected — {cột: simpleString}; name — tên bảng
    Output: None
    Ví dụ : _require_types(products, PRODUCT_TYPES, "slv_pn_products")
    """
    got = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    missing = [col for col in expected if col not in got]
    bad = [f"{col}: {got[col]}" for col in expected if col in got and got[col] != expected[col]]
    if missing or bad:
        raise ConfigError(f"{name}: thiếu {missing}; lệch kiểu {bad}")


def _spec():
    return {
        "trg_tbl": "slv_pn_hotel_room_offer",
        "inputs": {"room_types": "slv_pn_hotel_room_types"},
        "required_inputs": ("room_types",),
        "block_checks": (("is_bookable_null", "is_bookable IS NULL"),),
        "pk": ("room_type_id",),
        "deleted_flag": "_is_deleted",
        "write_mode": "MERGE",
    }


def build(ctx):
    rooms = ctx.input("room_types")
    _require_types(rooms, ROOM_TYPES, "slv_pn_hotel_room_types")
    ok = (F.col("status") == F.lit("Active")) & (F.col("available_rooms") > 0) & (F.col("deleted") == F.lit(False))
    rows = rooms.select(
        F.col("id").alias("room_type_id"),
        "status", "available_rooms", "max_occupancy", "adult_capacity", "child_capacity", "view_type", "deleted",
        F.coalesce(ok, F.lit(False)).alias("is_bookable"),
        F.col("deleted").alias(FLAG),
    )
    ctx.warn("room_status_null", rows.where(F.col("status").isNull()).count())
    return rows.select(*OUT_COLS, FLAG)


SPEC = NodeSpec(
    trg_tbl="slv_pn_hotel_room_offer",
    inputs={"room_types": "slv_pn_hotel_room_types"},
    pk=("room_type_id",),
    build=build,
    required_inputs=("room_types",),
    block_checks=(("is_bookable_null", "is_bookable IS NULL"),),
    deleted_flag=FLAG,
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
